In [2]:
import os
import numpy as np
from scipy.special import expit, logit
np.set_printoptions(linewidth=100)

np.random.seed(123)
n = 1234567
p = 50
x = np.random.normal(size=(n, p))

beta1 = np.random.normal(scale=1.0, size=p)
prob1 = expit(-2.718 + x.dot(beta1))  # p = 1 / (1 + exp(-x * beta))
y1 = np.random.binomial(1, prob1)

beta2 = np.random.normal(scale=0.5, size=p)
lam2 = np.exp(x.dot(beta2))  # lambda = exp(x * beta)
y2 = np.random.poisson(lam=lam2)

dat = np.hstack((y1.reshape(n, 1), y2.reshape(n, 1), x))
if not os.path.exists("data"):
    os.makedirs("data", exist_ok=True)
np.savetxt("data/sim_data.txt", dat, fmt="%f", delimiter=";", newline="\n",
    header="Y1\tY2\t" + "\t".join([f"V{i + 1}" for i in range(p)]), comments="# These are comments\n# and you should skip them\n")

In [3]:
# 此处插入代码
with open("data/sim_data.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())

# These are comments
# and you should skip them
Y1	Y2	V1	V2	V3	V4	V5	V6	V7	V8	V9	V10	V11	V12	V13	V14	V15	V16	V17	V18	V19	V20	V21	V22	V23	V24	V25	V26	V27	V28	V29	V30	V31	V32	V33	V34	V35	V36	V37	V38	V39	V40	V41	V42	V43	V44	V45	V46	V47	V48	V49	V50
1.000000;0.000000;-1.085631;0.997345;0.282978;-1.506295;-0.578600;1.651437;-2.426679;-0.428913;1.265936;-0.866740;-0.678886;-0.094709;1.491390;-0.638902;-0.443982;-0.434351;2.205930;2.186786;1.004054;0.386186;0.737369;1.490732;-0.935834;1.175829;-1.253881;-0.637752;0.907105;-1.428681;-0.140069;-0.861755;-0.255619;-2.798589;-1.771533;-0.699877;0.927462;-0.173636;0.002846;0.688223;-0.879536;0.283627;-0.805367;-1.727669;-0.390900;0.573806;0.338589;-0.011830;2.392365;0.412912;0.978736;2.238143
1.000000;2.000000;-1.294085;-1.038788;1.743712;-0.798063;0.029683;1.069316;0.890706;1.754886;1.495644;1.069393;-0.772709;0.794863;0.314272;-1.326265;1.417299;0.807237;0.045490;-0.233092;-1.198301;0.199524;0.468439;-0.831155;1.162204;-1.097203;-2.123100;1.03972

In [13]:
# 此处插入代码
import ray
import pandas as pd

ray.init()

chunk_refs = []
reader = pd.read_table(
    "data/sim_data.txt",
    delimiter=";",
    skiprows=3,
    header=None,
    chunksize=100000,
)

for chunk in reader:
    chunk_refs.append(ray.put(chunk.to_numpy(dtype=np.float64)))

print("number of chunks:", len(chunk_refs))
print("first ObjectRef:", chunk_refs[0])

2026-06-03 17:52:27,950	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


number of chunks: 13
first ObjectRef: ObjectRef(00ffffffffffffffffffffffffffffffffffffff010000000ee1f505)


In [5]:
# 此处插入代码
import time
import numpy as np
from scipy.special import expit
from scipy.optimize import fmin_l_bfgs_b

@ray.remote
def logit_chunk(dat, beta, need_hess=True):
    y = dat[:, 0]
    x0 = dat[:, 2:]

    X = np.empty((x0.shape[0], x0.shape[1] + 1), dtype=np.float64)
    X[:, 0] = 1.0
    X[:, 1:] = x0

    eta = X @ beta
    p_hat = expit(eta)
    loss = np.sum(-y * eta + np.logaddexp(0.0, eta))
    grad = X.T @ (p_hat - y)

    if need_hess:
        w = p_hat * (1.0 - p_hat)
        hess = X.T @ (X * w[:, None])
    else:
        hess = np.empty((0, 0))
    return loss, grad, hess, y.size


def logit_sum(beta, need_hess=True):
    ans = ray.get([logit_chunk.remote(ref, beta, need_hess) for ref in chunk_refs])
    loss = sum(a[0] for a in ans)
    grad = sum((a[1] for a in ans), start=np.zeros_like(beta))
    nobs = sum(a[3] for a in ans)
    if need_hess:
        hess = sum((a[2] for a in ans), start=np.zeros((beta.size, beta.size)))
    else:
        hess = None
    return loss, grad, hess, nobs


p_all = 51
beta_newton = np.zeros(p_all)
tol = 1e-6
max_iter = 30

st = time.time()
for it in range(max_iter):
    loss, grad, hess, nobs = logit_sum(beta_newton, need_hess=True)
    step = np.linalg.solve(hess + 1e-8 * np.eye(p_all), grad)

    alpha = 1.0
    while alpha > 1e-4:
        beta_try = beta_newton - alpha * step
        loss_try, _, _, _ = logit_sum(beta_try, need_hess=False)
        if loss_try <= loss:
            break
        alpha *= 0.5

    beta_newton = beta_try
    if np.linalg.norm(alpha * step) < tol:
        break

time_newton = time.time() - st
print("Newton iterations:", it + 1)
print("Newton time:", time_newton)
print("Newton beta:")
print(beta_newton)

def obj_grad(beta):
    loss, grad, _, _ = logit_sum(beta, need_hess=False)
    return loss, grad

st = time.time()
beta_lbfgs, f_min, info = fmin_l_bfgs_b(
    func=obj_grad,
    x0=np.zeros(p_all),
    maxiter=100,
    pgtol=1e-6,
)
time_lbfgs = time.time() - st

print("L-BFGS iterations:", info.get("nit"))
print("L-BFGS time:", time_lbfgs)
print("L-BFGS beta:")
print(beta_lbfgs)

Newton iterations: 9
Newton time: 15.373921871185303
Newton beta:
[-2.71401433  1.20619839 -0.64133165  0.9298867  -0.43226456  0.32472114  1.30381751 -0.4559375
  1.11217005  0.38065281  0.53073444 -0.47982553  1.39609521 -0.14809205  0.00382176  0.43072584
  0.21919829  1.84517604 -0.73457686 -1.0625383  -0.4464374   0.25138478  0.88208778 -2.00626937
  0.83569942 -0.67326866  1.63398641 -0.85893095 -0.76419692  2.47998678  0.1767243   0.41945422
 -0.59682832 -0.69250343  0.46309924  0.23270122 -2.09427042 -0.49284362 -0.67411165 -0.29742859
  0.35726432  0.41390704  0.51978605  1.52157982 -0.1984054  -0.077238   -0.09735507  0.91646183
 -0.02521668 -1.39049315  0.61383634]
L-BFGS iterations: 14
L-BFGS time: 5.248133897781372
L-BFGS beta:
[-2.71401369  1.20619817 -0.64133158  0.92988659 -0.43226467  0.32472109  1.30381741 -0.45593753
  1.11216975  0.3806526   0.53073449 -0.47982537  1.39609481 -0.148092    0.00382163  0.43072593
  0.21919814  1.84517576 -0.7345769  -1.06253808 -0.446

In [6]:
# 此处插入代码
@ray.remote
def acc_chunk(dat, beta):
    y = dat[:, 0]
    x0 = dat[:, 2:]

    X = np.empty((x0.shape[0], x0.shape[1] + 1), dtype=np.float64)
    X[:, 0] = 1.0
    X[:, 1:] = x0

    prob = expit(X @ beta)
    label = (prob >= 0.5).astype(np.float64)
    return np.sum(label == y), y.size

acc_parts = ray.get([acc_chunk.remote(ref, beta_lbfgs) for ref in chunk_refs])
correct = sum(a[0] for a in acc_parts)
total = sum(a[1] for a in acc_parts)
accuracy = correct / total

print("accuracy:", accuracy)

accuracy: 0.9244382848399479


考虑在 $Y_2$ 与 $X$ 之间建立 Poisson 回归模型，具体形式为：给定第 $i$ 个自变量观测 $x_i$，假定因变量 $Y_{2i}$ 服从均值为 $\lambda_i$ 的 Poisson 分布，其中 $\lambda_i=e^{x_i'\beta}$，$\beta$ 是回归系数。请推导写出 Poisson 回归模型的似然函数和它关于 $\beta$ 的梯度。

对第 $i$ 个观测，Poisson 回归假定

$$
Y_{2i}\mid x_i\sim \text{Poisson}(\lambda_i),\qquad \lambda_i=\exp(x_i'\beta).
$$

因此单个观测的概率质量函数为

$$
P(Y_{2i}=y_i\mid x_i)
=\frac{\exp(-\lambda_i)\lambda_i^{y_i}}{y_i!}
=\frac{\exp[-\exp(x_i'\beta)]\exp(y_i x_i'\beta)}{y_i!}.
$$

在样本独立的条件下，整体似然函数为

$$
L(\beta)=\prod_{i=1}^n
\frac{\exp[-\exp(x_i'\beta)]\exp(y_i x_i'\beta)}{y_i!}.
$$

对应的对数似然函数为

$$
\ell(\beta)
=\sum_{i=1}^n\left[y_i x_i'\beta-\exp(x_i'\beta)-\log(y_i!)\right].
$$

对 $\beta$ 求梯度得到

$$
\nabla \ell(\beta)
=\sum_{i=1}^n x_i\left[y_i-\exp(x_i'\beta)\right].
$$

令 $\lambda=(\lambda_1,\ldots,\lambda_n)'$，矩阵形式为

$$
\nabla \ell(\beta)=X'(y-\lambda).
$$

如果把目标函数写成负对数似然并进行最小化，则梯度为

$$
\nabla[-\ell(\beta)]=X'(\lambda-y).
$$

利用 ADMM 算法求解 Lasso 问题
$$\frac{1}{2}\Vert y-X\beta\Vert^2+\lambda \Vert \beta\Vert_1,$$
并将其封装成一个函数：

```python
admm_lasso(X, y, lam, rho=1.0, maxit=10000, eps=1e-3, verbose=0)
```

1. 其中 `X` 是 $n\times p$ 的自变量矩阵，`y` 是 $n\times 1$ 的因变量向量，`lam` 是惩罚项参数 $\lambda$，`rho` 是 ADMM 算法的 $\rho$ 参数，`maxit` 是最大迭代次数，`eps` 是 ADMM 收敛的残差临界值，`verbose` 表示是否输出迭代信息，如果 $>0$，则每隔 1000 次迭代打印出当前的两类残差，如果 $\le 0$ 则不输出任何信息。
2. 参考 `lec12-admm-lad.ipynb` 中的 Cholesky 分解方法，只对矩阵进行一次分解，从而在每次迭代中高效地求解线性方程组。
3. 函数需返回两个量，第一个表示实际使用的迭代次数，第二个表示估计的回归系数。

In [7]:
# 此处插入代码
import numpy as np
from scipy.linalg import cho_factor, cho_solve

def soft_threshold(a, kappa):
    return np.sign(a) * np.maximum(np.abs(a) - kappa, 0.0)


def admm_lasso(X, y, lam, rho=1.0, maxit=10000, eps=1e-3, verbose=0):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    n, p = X.shape

    XtX = X.T @ X
    Xty = X.T @ y
    cf = cho_factor(XtX + rho * np.eye(p), lower=True, check_finite=False)

    beta = np.zeros(p)
    z = np.zeros(p)
    u = np.zeros(p)

    for it in range(1, maxit + 1):
        beta = cho_solve(cf, Xty + rho * (z - u), check_finite=False)

        z_old = z.copy()
        z = soft_threshold(beta + u, lam / rho)
        u = u + beta - z

        r_norm = np.linalg.norm(beta - z)
        s_norm = rho * np.linalg.norm(z - z_old)

        if verbose > 0 and (it == 1 or it % 1000 == 0):
            print(f"iter={it}, primal={r_norm:.6e}, dual={s_norm:.6e}")

        if r_norm < eps and s_norm < eps:
            break

    return it, beta

In [8]:
np.random.seed(123)
n = 1000
p = 30
nz = 20
Xtrain = np.random.normal(size=(n, p))
beta = np.random.normal(size=nz)
beta = np.concatenate((beta, np.zeros(p - nz)))
ytrain = Xtrain.dot(beta) + np.random.normal(size=n)
beta

array([-1.05417044, -0.78301134,  1.82790084,  1.7468072 ,  1.3282585 , -0.43277314, -0.6686141 ,
       -0.47208845,  1.05554064,  0.67905585,  0.14814832,  1.04294573,  0.28718991,  1.55577283,
        0.97031604,  0.39737593,  1.15394013, -0.00333042,  1.30948521, -0.90230241,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

In [9]:
admm_lasso(Xtrain, ytrain, lam=0.1 * n, maxit=10000, eps=1e-3, verbose=1)

iter=1, primal=4.571809e+00, dual=0.000000e+00
iter=1000, primal=6.801351e-02, dual=8.768316e-06
iter=2000, primal=1.903375e-02, dual=2.632103e-06
iter=3000, primal=7.489876e-03, dual=1.010766e-06
iter=4000, primal=2.980396e-03, dual=3.922571e-07
iter=5000, primal=1.199739e-03, dual=1.538648e-07


(5202,
 array([-9.88446192e-01, -7.29951991e-01,  1.72843395e+00,  1.66188615e+00,  1.18779108e+00,
        -1.94466297e-01, -5.94711189e-01, -3.91430856e-01,  1.01063023e+00,  5.73786673e-01,
         3.36364138e-02,  9.31135970e-01,  2.21897026e-01,  1.51032137e+00,  9.07779872e-01,
         2.93449914e-01,  1.08151311e+00, -2.97145094e-04,  1.17431918e+00, -7.88572868e-01,
        -1.56776389e-04,  3.81700889e-04, -4.71138798e-04, -2.07577854e-04, -3.67392622e-04,
        -3.15013324e-04, -8.35414719e-06,  5.02959595e-06,  4.88460731e-04, -5.39329955e-05]))

In [10]:
admm_lasso(Xtrain, ytrain, lam=0.01 * n, maxit=10000, eps=1e-3, verbose=0)

(1287,
 array([-1.07555904e+00, -8.14460224e-01,  1.79118556e+00,  1.72909346e+00,  1.27448621e+00,
        -3.06897473e-01, -6.69469287e-01, -4.73021701e-01,  1.09124222e+00,  6.69340756e-01,
         1.24876014e-01,  1.02527211e+00,  3.02106479e-01,  1.58722372e+00,  9.68663224e-01,
         3.84937457e-01,  1.15919477e+00, -3.76698667e-02,  1.27397237e+00, -9.01267833e-01,
        -7.42409908e-03,  1.91910835e-02, -6.06421740e-02, -2.57202269e-02, -1.93040114e-02,
        -3.33947154e-02,  9.99301503e-04, -1.52979627e-02,  2.22508621e-02, -2.01260868e-02]))